# YOLO

**YOLO (You Only Look Once)** is a real-time object detection algorithm that treats detection as a single regression problem. Instead of scanning different parts of the image multiple times, YOLO processes the entire image in one forward pass of a neural network, making it extremely fast and suitable for real-time applications.

- YOLO was first introduced in 2016 and later significantly improved by the Ultralytics team. It is widely used in computer vision because it provides a strong balance between speed and accuracy.

- YOLO is especially useful for building real-time, production-ready computer vision systems and is widely adopted in both research and industry applications.
---

## What YOLO Can Help You With

1. **Object Detection**
   Detect and localize objects such as people, vehicles, animals, and products.

2. **Instance Segmentation**
   Detect objects and generate pixel-level masks for each instance.

3. **Image Classification**
   Classify entire images into predefined categories.

4. **Object Tracking**
   Track detected objects across video frames.

5. **Pose Estimation**
   Detect human keypoints for body posture and movement analysis.

6. **Oriented Bounding Boxes (OBB)**
   Detect rotated objects, useful in aerial and satellite imagery.

7. **Real-Time Video Analytics**
   Deploy on edge devices for surveillance, smart cities, robotics, and autonomous systems.

---

## Image Annotation

**Image Annotation** is the process of labeling images to make them understandable for machine learning models. In computer vision, annotated images serve as the ground truth data that models learn from during training.

Annotation helps models recognize patterns, objects, boundaries, keypoints, and other visual features. Without properly labeled data, even the most advanced deep learning models cannot learn effectively.


### Why Image Annotation is Important

1. **Model Training**
   Provides supervised learning data for detection, segmentation, and classification models.

2. **Improves Accuracy**
   High-quality annotations directly improve model performance.

3. **Enables Complex Tasks**
   Supports advanced applications like pose estimation, medical imaging analysis, and autonomous vehicles.

4. **Model Evaluation**
   Used to validate and benchmark computer vision systems.


### Image Annotation Tools
   - [CVAT](https://www.cvat.ai/)

---

## The Official Documentation
  - https://github.com/ultralytics/ultralytics

## Installation

In [ ]:
!pip install ultralytics

In [2]:
from ultralytics import YOLO

---

## How to Install Specific Dataset

## Object Detection

### Required packages

#### Install Kaggle API
```
pip install kaggle
```
#### Download openimages list of classes
```
pip install openimages
```
#### Download OIDv6_ToolKit for dataset management
```
git clone https://github.com/EscVM/OIDv6_ToolKit.git
cd OIDv4_ToolKit
pip install -r requirements.txt
```
#### Downloading the dataset
```
python main.py downloader --classes Tea --type_csv train --multiclass 1 --limit 800
```

---

## Yolo with webcam

### On CPU

In [ ]:
from ultralytics import YOLO
import torch
import cv2
import cvzone
import time

# ---------------------------
# Force CPU
# ---------------------------
device = "cpu"

print("Running on CPU")
print("Press 'Q' to quit...\n")

# ---------------------------
# Load YOLO Model on CPU
# ---------------------------
model = YOLO("models/yolo26n.pt")
model.to(device)

# ---------------------------
# Initialize Webcam
# ---------------------------
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

cap.set(3, 1280)
cap.set(4, 720)

prev_time = 0

# ---------------------------
# Main Loop
# ---------------------------
try:
    while True:
        success, frame = cap.read()

        if not success:
            print("Failed to read frame.")
            break

        # ---------------------------
        # YOLO Inference (CPU)
        # ---------------------------
        results = model(
            frame,
            stream=True,
            device="cpu",
            imgsz=640,
            conf=0.5
        )

        for result in results:
            if result.boxes is None:
                continue

            for box in result.boxes:
                # Bounding box
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (255, 0, 0), 2)

                # Confidence
                conf = f"{box.conf[0].item():.2f}"

                # Class name
                cls_id = int(box.cls[0].item())
                cls = result.names[cls_id]

                cvzone.putTextRect(
                    frame,
                    f"{cls} {conf}",
                    (max(0, x1), max(35, y1)),
                    scale=1,
                    thickness=1
                )

        # ---------------------------
        # FPS Counter
        # ---------------------------
        current_time = time.time()
        fps = 1 / (current_time - prev_time)
        prev_time = current_time

        cvzone.putTextRect(
            frame,
            f"FPS: {int(fps)}",
            (20, 50),
            scale=2,
            thickness=2
        )

        cv2.imshow("YOLO CPU Detection", frame)

        # Press Q to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Exiting...")
            break

except Exception as e:
    print("Error occurred:", e)

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Camera released. Program closed safely.")

### On Gpu

In [ ]:
from ultralytics import YOLO
import torch
import cv2
import cvzone
import time

# ---------------------------
# GPU Optimization
# ---------------------------
torch.backends.cudnn.benchmark = True

if not torch.cuda.is_available():
    print("CUDA is not available. Exiting...")
    exit()

device = 0  # First GPU

# ---------------------------
# Load YOLO Model on GPU
# ---------------------------
model = YOLO("models/yolo26l.pt")

print("Running on GPU:", torch.cuda.get_device_name(0))
print("Press 'Q' to quit...\n")

# ---------------------------
# Initialize Webcam
# ---------------------------
cap = cv2.VideoCapture("data_for_object_detection/inputs/test.mp4")

if not cap.isOpened():
    print("Error: Could not open webcam.")
    exit()

cap.set(3, 1280)
cap.set(4, 720)

prev_time = 0

# ---------------------------
# Main Loop
# ---------------------------
try:
    while True:
        success, frame = cap.read()

        if not success:
            print("Failed to read frame.")
            break

        # ---------------------------
        # YOLO Inference (GPU + FP16)
        # ---------------------------
        results = model(
            frame,
            stream=True,
            device=device,
            imgsz=640,
            half=True,
            conf=0.5
        )

        for result in results:
            if result.boxes is None:
                continue

            for box in result.boxes:
                # Bounding box
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                cv2.rectangle(frame, (x1, y1), (x2, y2),
                              (255, 0, 0), 2)

                # Confidence
                conf = f"{box.conf[0].item():.2f}"

                # Class name
                cls_id = int(box.cls[0].item())
                cls = result.names[cls_id]

                cvzone.putTextRect(
                    frame,
                    f"{cls} {conf}",
                    (max(0, x1), max(35, y1)),
                    scale=1,
                    thickness=1
                )

        # ---------------------------
        # FPS Counter
        # ---------------------------
        current_time = time.time()
        fps = 1 / (current_time - prev_time)
        prev_time = current_time

        cvzone.putTextRect(
            frame,
            f"FPS: {int(fps)}",
            (20, 50),
            scale=2,
            thickness=2
        )

        cv2.imshow("YOLO GPU Detection", frame)

        # Press Q to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Exiting...")
            break

except Exception as e:
    print("Error occurred:", e)

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Camera released. Program closed safely.")

---

## Object Detection

### Load Model

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kylegraupe/wind-turbine-image-dataset-for-computer-vision")

print("Path to dataset files:", path)

In [ ]:
# Load a pretrained YOLO26n model
model = YOLO("models/yolo26n.pt")

### Train on custom images

In [ ]:
train_results = model.train(
    data="data_for_object_detection.yaml",  # Path to dataset configuration file
    epochs=1,  # Number of training epochs
    imgsz=640,  # Image size for training
)

### Model Evaluation

In [ ]:
# Evaluate the model's performance on the validation set
metrics = model.val()

### Model Prediction

In [ ]:
# Perform object detection on an image
results = model("./test.jpg")  # Predict on an image
results[0].show()  # Display results

### Perform object detection on a video

In [ ]:
import cv2

# Open the video file
video_path = "data_for_object_detection/inputs/test.mp4"  # Change path if needed
cap = cv2.VideoCapture(video_path)

# Check if video opened successfully
if not cap.isOpened():
    print(f"Error opening video file: {video_path}")
else:
    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Define output video writer (optional)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter('data_for_object_detection/outputs/output_detected.mp4', fourcc, fps, (width, height))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Run YOLO detection on the frame
        results = model(frame)
        # Visualize results on the frame
        annotated_frame = results[0].plot()

        # Show the frame (optional, comment out if running headless)
        cv2.imshow('YOLO Detection', annotated_frame)

        # Write the frame to output video
        out.write(annotated_frame)

        # Press 'q' to exit early
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()
    print("Detection complete. Output saved as output_detected.mp4")

### Exporting Model

In [ ]:
# Export the model to ONNX format for deployment
path = model.export(format="onnx")  # Returns the path to the exported model

---

## Image Segmentation

### Loading Dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("quadeer15sh/augmented-forest-segmentation")

print("Path to dataset files:", path)

D:\Programming\Material\Computer Vision\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 172M/172M [01:42<00:00, 1.76MB/s] 

Extracting files...


Path to dataset files: C:\Users\Makrious\.cache\kagglehub\datasets\quadeer15sh\augmented-forest-segmentation\versions\2


### Convert Binary Masks to Polygons (For YOLO Segmentation Training)

In [7]:
import os
import cv2

input_dir = './data_for_image_segmentation/masks'
output_dir = './data_for_image_segmentation/labels'

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

print(f"Input directory: {input_dir}")
print(f"Output directory: {output_dir}")

mask_files = os.listdir(input_dir)
print(f"Found {len(mask_files)} mask files.")

for j in mask_files:
    image_path = os.path.join(input_dir, j)
    print(f"Processing: {image_path}")
    mask = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(f"Warning: Could not read mask file {image_path}")
        continue
    _, mask = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)

    H, W = mask.shape
    contours, hierarchy = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print(f"Found {len(contours)} contours.")

    polygons = []
    for cnt in contours:
        if cv2.contourArea(cnt) > 10:  # Lowered threshold
            polygon = []
            for point in cnt:
                x, y = point[0]
                polygon.append(x / W)
                polygon.append(y / H)
            polygons.append(polygon)
    print(f"Polygons to write: {len(polygons)}")

    output_file = '{}.txt'.format(os.path.join(output_dir, j)[:-4])
    if polygons:
        with open(output_file, 'w') as f:
            for polygon in polygons:
                for p_, p in enumerate(polygon):
                    if p_ == len(polygon) - 1:
                        f.write('{}\n'.format(p))
                    elif p_ == 0:
                        f.write('0 {} '.format(p))
                    else:
                        f.write('{} '.format(p))
            print(f"Wrote file: {output_file}")
    else:
        print(f"No polygons found for {j}, skipping file write.")

Input directory: ./data_for_image_segmentation/masks
Output directory: ./data_for_image_segmentation/labels
Found 5108 mask files.
Processing: ./data_for_image_segmentation/masks\10452_mask_08.jpg
Found 60 contours.
Polygons to write: 1
Wrote file: ./data_for_image_segmentation/labels\10452_mask_08.txt
Processing: ./data_for_image_segmentation/masks\10452_mask_18.jpg
Found 12 contours.
Polygons to write: 1
Wrote file: ./data_for_image_segmentation/labels\10452_mask_18.txt
Processing: ./data_for_image_segmentation/masks\111335_mask_00.jpg
Found 153 contours.
Polygons to write: 1
Wrote file: ./data_for_image_segmentation/labels\111335_mask_00.txt
Processing: ./data_for_image_segmentation/masks\111335_mask_01.jpg
Found 149 contours.
Polygons to write: 2
Wrote file: ./data_for_image_segmentation/labels\111335_mask_01.txt
Processing: ./data_for_image_segmentation/masks\111335_mask_02.jpg
Found 144 contours.
Polygons to write: 2
Wrote file: ./data_for_image_segmentation/labels\111335_mask_02

### Load Model and Train

In [14]:
model = YOLO("models/YOLO26s-seg.pt")

# Use the correct relative path for config.yaml
model.train(data="data_for_image_segmentation/config.yaml", epochs=1, imgsz=256)

Ultralytics 8.4.14  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_for_image_segmentation/config.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/YOLO26s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

KeyboardInterrupt: 

### Predict on an image

In [ ]:
from ultralytics import YOLO

import cv2


model_path = '...'

image_path = '...'

img = cv2.imread(image_path)
H, W, _ = img.shape

model = YOLO(model_path)

results = model(img)

for result in results:
    for j, mask in enumerate(result.masks.data):
        mask = mask.numpy() * 255
        mask = cv2.resize(mask, (W, H))

        cv2.imwrite('./output.png', mask)

---